# Stage 2 – Customer Clustering Analysis

This notebook is prepared for the ACS Stage 2 clustering task.

**Role:** Data Analyst – Clustering Analysis  
**Clustering variables:** `tenure` and `MonthlyCharges`  
**Method:** K-Means  
**Optimal K used for the project:** 4 (based on the observed Elbow Method)


## Step 1 – Upload your CSV file

Run the cell below, then choose your dataset file, such as **`churn_clustering_ready (2).csv`**.


In [ ]:
from google.colab import files
uploaded = files.upload()

print('Uploaded file(s):')
for name in uploaded.keys():
    print('-', name)


In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

# Automatically find the uploaded CSV file
csv_files = [f for f in os.listdir('/content') if f.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('No CSV file was uploaded. Please run the upload cell first.')

csv_path = '/content/' + csv_files[0]
df = pd.read_csv(csv_path)

print('Dataset loaded:', os.path.basename(csv_path))
print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df.head())


## Step 2 – Check the clustering columns

We use **tenure** and **MonthlyCharges** to create the customer segments. Churn is not used to create the clusters; it is used later to interpret the clusters.

In [ ]:
required = ['tenure', 'MonthlyCharges']
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f'Missing required column(s): {missing}. Available columns are: {list(df.columns)}')

features = df[required].copy()
print(features.describe())
print('\nMissing values:')
print(features.isna().sum())


## Step 3 – Prepare the data

The supplied clustering variables are already standardised in the project dataset, so no additional scaling is applied here.

We split the data into **80% training** and **20% testing** sets.

In [ ]:
X = features.dropna().copy()

X_train, X_test = train_test_split(
    X, test_size=0.20, random_state=42
)

print(f'Training records: {len(X_train)}')
print(f'Testing records: {len(X_test)}')


## Step 4 – Elbow Method

The Elbow Method helps us choose a reasonable number of clusters by comparing K-Means inertia for different values of K.

In [ ]:
inertias = []
k_values = range(1, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_train)
    inertias.append(model.inertia_)

elbow_df = pd.DataFrame({'K': list(k_values), 'Inertia': inertias})
display(elbow_df)

plt.figure(figsize=(8, 5))
plt.plot(elbow_df['K'], elbow_df['Inertia'], marker='o')
plt.title('Elbow Method for K-Means')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(list(k_values))
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/elbow_method.png', dpi=200)
plt.show()


### Project choice: K = 4

For this project, **K = 4** is used as the working solution based on the observed elbow in the plotted curve. This is a practical selection rather than claiming that K = 4 is the only possible answer.

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans.fit(X_train)

# Predict clusters for all available rows used for clustering
cluster_labels = kmeans.predict(X)

clustered = X.copy()
clustered['Cluster'] = cluster_labels

print('K-Means model trained with K =', optimal_k)
display(clustered.head())


## Step 5 – Cluster visualisation

In [ ]:
plt.figure(figsize=(9, 6))
for cluster_id in sorted(clustered['Cluster'].unique()):
    subset = clustered[clustered['Cluster'] == cluster_id]
    plt.scatter(subset['tenure'], subset['MonthlyCharges'], s=18, alpha=0.6, label=f'Cluster {cluster_id}')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=180, label='Centroids')
plt.title('Customer Segments from K-Means Clustering')
plt.xlabel('Tenure (standardised)')
plt.ylabel('Monthly Charges (standardised)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/cluster_visualisation.png', dpi=200)
plt.show()


## Step 6 – Cluster summary

The following table describes the size and average values of each cluster.

In [ ]:
summary = clustered.groupby('Cluster').agg(
    Customers=('Cluster', 'size'),
    Avg_Tenure=('tenure', 'mean'),
    Avg_MonthlyCharges=('MonthlyCharges', 'mean')
).reset_index()

summary['Tenure_Level'] = np.where(summary['Avg_Tenure'] >= 0, 'Relatively high', 'Relatively low')
summary['MonthlyCharge_Level'] = np.where(summary['Avg_MonthlyCharges'] >= 0, 'Relatively high', 'Relatively low')

display(summary)
summary.to_csv('/content/cluster_summary.csv', index=False)


## Step 7 – Optional churn interpretation

If the dataset contains a churn column, we can compare churn rates **after** clustering. Churn is not used as an input to K-Means.

In [ ]:
churn_col = next((c for c in df.columns if c.lower() in ['churn', 'churn_value']), None)

if churn_col is not None:
    temp = df.loc[X.index].copy()
    temp['Cluster'] = cluster_labels
    churn = temp.groupby('Cluster')[churn_col].agg(['count', 'mean']).reset_index()
    churn['Churn_Rate_Percent'] = churn['mean'] * 100
    display(churn[['Cluster', 'count', 'Churn_Rate_Percent']])
    churn.to_csv('/content/churn_by_cluster.csv', index=False)
else:
    print('No standard churn column was found. Skipping churn interpretation.')


## Step 8 – Save project files

This cell saves the trained model and key outputs so they can be shared with the Project Manager.

In [ ]:
joblib.dump(kmeans, '/content/kmeans_model.pkl')
clustered.to_csv('/content/clustered_dataset.csv', index=False)
elbow_df.to_csv('/content/elbow_values.csv', index=False)

print('Saved:')
for f in [
    'elbow_method.png',
    'cluster_visualisation.png',
    'cluster_summary.csv',
    'clustered_dataset.csv',
    'elbow_values.csv',
    'kmeans_model.pkl'
]:
    print('✓', f)


## Presentation summary

Use this simple explanation when presenting:

1. I used customer **tenure** and **monthly charges** for clustering.
2. I checked the data and used an 80/20 training/testing split.
3. I used the **Elbow Method** to compare different numbers of clusters.
4. I selected **K = 4** as a reasonable working solution based on the observed elbow.
5. I trained a **K-Means** model and visualised the four customer segments.
6. I interpreted the clusters using relative tenure and monthly-charge levels.
7. I used churn only as a post-clustering interpretation measure, not as a clustering input.